# 10 - Base-model choice: memorization probes

Two parts, two runtimes. Everything is kept in the private Hub repo `Tsomaros/llm-movies-raw`; nothing stays on the Colab disk.

- **Part A (CPU runtime), once.** Rebuilds `movies.db` from the raw shards and uploads it as `db/movies.db`. If the repo already holds it, part A does nothing.
- **Part B (T4 GPU runtime).** Downloads `db/movies.db` and runs the director probe (150 films per release quarter, 2019Q1 to 2026Q2, at least 100 votes) and the revenue probe (300 validation films) for each candidate base model. After each model it uploads the CSVs and the summary to `model_choice/` in the repo.

**To resume part B: rerun it.** A model already `ok` in the repo's `summary.json` is skipped; a failed one is tried again. Candidates are listed in `llm_movies/eval/model_choice.py`; the gated ones (Llama 3.2 and Gemma 3) come after the open ones, and the conditional Gemma 4 E2B is last; the gated ones need the licence accepted on huggingface.co first (see `notebooks/README.md`).

`HF_TOKEN` needs write access to the raw repo and read access to the gated models. Each part's last cell is **one chain** (`&&`): a failed step stops the rest.

## Setup

Run these cells top to bottom. They are the same in all notebooks. **After a runtime change, run them again** (the new machine starts empty). The Python packages are installed by the part you run (A or B), because the two runtimes need different PyTorch builds.

In [ ]:
# clone the repo the first time, update it after that
!test -d /content/LLM_Movies_Model/.git && git -C /content/LLM_Movies_Model pull || git clone https://github.com/Tsomaros/LLM_Movies_Model /content/LLM_Movies_Model
%cd /content/LLM_Movies_Model

In [ ]:
!pip install -q uv

## Secrets and environment

Reads `HF_TOKEN` from Colab Secrets, with notebook access switched on. It is never printed. `UV_FROZEN` stops later `uv run` calls from re-resolving the environment.

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["UV_FROZEN"] = "1"

## Part A - build and store movies.db (CPU runtime)

Skip this part when `db/movies.db` is already in the repo (the command checks and says so). It downloads the TMDB and Wikidata shards (several GB) and builds on the local disk, so use the CPU runtime. Part A needs no PyTorch: run the sync cell, then the command cell.

In [ ]:
!uv sync --frozen

In [ ]:
!uv run python -m llm_movies.data.db_snapshot ensure --raw-dir /content/raw --out /content/build/movies.db --repo Tsomaros/llm-movies-raw

## Part B - probe the candidate models (T4 GPU runtime)

Switch the runtime to **T4 GPU** (Runtime > Change runtime type), then run the **Setup** and **Secrets** cells again (not part A), then the two cells below. The first installs the `train` extra and CUDA PyTorch (a few minutes; the download is several GB). The second is the probe; without a GPU it stops with a message instead of running for hours on the CPU.

In [ ]:
!uv sync --frozen --extra train --extra cu132

In [ ]:
!uv run python -m llm_movies.data.db_snapshot fetch --out /content/build/movies.db --repo Tsomaros/llm-movies-raw && uv run python -m llm_movies.eval.model_choice run --db /content/build/movies.db --out /content/model_choice --repo Tsomaros/llm-movies-raw